# Notebook 09 - Imported Component Mixed System

This notebook demonstrates the mixed-system boundary: a custom STEP/STL asset has its own local CAD coordinates, Tuba places that asset in global model coordinates, a programmatic pipe connects to the transformed port, and the existing web-scene viewer renders the relationship. The mixed Code_Aster export remains a handoff until a real mixed solve/import path is proven.

In [ ]:
from pathlib import Path
import sys
from IPython.display import HTML, display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "examples").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from examples.imported_component_mixed_system import publish_viewer_bundle, run_demo

WORK_DIR = REPO_ROOT / "imported_component_mixed_demo"
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Local CAD coordinates: the nozzle port is at local [0, 0, 0].
# The equipment body sits mostly in +X local space behind that nozzle.
LOCAL_BOUNDS = [-0.05, -0.18, -0.18, 0.45, 0.18, 0.18]
STL_PATH = WORK_DIR / "local_equipment.stl"

import trimesh

box = trimesh.creation.box(extents=(0.50, 0.36, 0.36))
box.apply_translation((0.20, 0.0, 0.0))
box.export(STL_PATH)

STEP_PATH = WORK_DIR / "local_equipment.step"
try:
    import gmsh

    gmsh.initialize()
    try:
        gmsh.model.add("local_equipment_step")
        gmsh.model.occ.addBox(*LOCAL_BOUNDS[:3], *(LOCAL_BOUNDS[i + 3] - LOCAL_BOUNDS[i] for i in range(3)))
        gmsh.model.occ.synchronize()
        gmsh.write(str(STEP_PATH))
    finally:
        gmsh.finalize()
    SOURCE_PATH = STEP_PATH
except Exception:
    SOURCE_PATH = STL_PATH

SOURCE_PATH

The placement below maps asset-local coordinates into the Tuba global coordinate system:

`global_point = placement.origin + placement.rotation @ local_point`

Changing `ASSET_ORIGIN_GLOBAL` moves the imported component and its confirmed port. The programmatic pipe is built to the transformed global port position.

In [ ]:
ASSET_ORIGIN_GLOBAL = [1.2, 0.45, 0.0]
ASSET_ROTATION_QWXYZ = [1.0, 0.0, 0.0, 0.0]

summary = run_demo(
    SOURCE_PATH,
    output_root=WORK_DIR / "run",
    export_study=False,
    asset_origin=ASSET_ORIGIN_GLOBAL,
    asset_rotation=ASSET_ROTATION_QWXYZ,
)
viewer_bundle = publish_viewer_bundle(summary["scene_dir"])
summary

In [ ]:
import tuba

model = tuba.Model.from_json(summary["model_path"])
asset = model.cad_assets["cad_asset_custom_equipment"]
port = model.ports["port_equipment_nozzle_a"]
coupling = model.couplings["coupling_pipe_to_equipment_a"]

{
    "asset_local_to_global_placement": asset.placement,
    "port_local_position": port.metadata["local_position"],
    "port_global_position": list(port.position),
    "pipe_endpoint_node": coupling.source_node.id,
    "coupling": coupling.to_dict(),
}

Start the viewer from the repo root if it is not already running:

`cd viewer && npm.cmd run dev -- --host 127.0.0.1`

Then open the bundle below.

In [ ]:
import os

VIEWER_BASE_URL = os.environ.get("TUBA_VIEWER_URL", "http://127.0.0.1:5173").rstrip("/")
VIEWER_URL = f"{VIEWER_BASE_URL}/?bundle=/imported_component_mixed_demo"
display(HTML(f'<p><a href="{VIEWER_URL}" target="_blank">Open imported component review scene</a></p>'
             f'<iframe src="{VIEWER_URL}" width="100%" height="720" style="border:0; min-height:520px;"></iframe>'))